In [1]:
# import basic packages
import numpy as np

import pandas as pd

import warnings
warnings.filterwarnings(action = "ignore")

In [3]:
# load the dataset
# lines = pd.read_csv("movie_lines.tsv", sep="\t", encoding="latin-1")
# conversations = pd.read.csv("movie_conversations.tsv", sep="\t", encoding="latin-1")

In [19]:
lines = pd.read_csv("movie_lines.tsv", sep="\t", header=None,
                    names=["line_id", "user_id", "movie_id", "name", "text"],
                    encoding="latin-1", on_bad_lines='skip')

conversations = pd.read_csv(
    "movie_conversations.tsv",
    sep="\t",
    header=None,
    encoding="latin-1",
    engine="python"
)

# Your dataset has messy rows → pandas fails → skip bad lines to fix

In [23]:
conversations["conversation"] = conversations[3]

In [34]:
# create line dictionary
#connects line IDs to real dialogue text, so the chatbot can understand and respond.
line_dict = dict(zip(lines["line_id"], lines["text"]))

In [35]:
print(conversations.head())

    0   1   2                              3                   conversation
0  u0  u2  m0  ['L194' 'L195' 'L196' 'L197']  ['L194' 'L195' 'L196' 'L197']
1  u0  u2  m0                ['L198' 'L199']                ['L198' 'L199']
2  u0  u2  m0  ['L200' 'L201' 'L202' 'L203']  ['L200' 'L201' 'L202' 'L203']
3  u0  u2  m0         ['L204' 'L205' 'L206']         ['L204' 'L205' 'L206']
4  u0  u2  m0                ['L207' 'L208']                ['L207' 'L208']


In [36]:
print(conversations["conversation"].iloc[0])

['L194' 'L195' 'L196' 'L197']


In [37]:
pairs = []

for conv in conversations["conversation"]:
    
    conv = str(conv)
    
    # Clean string properly
    conv = conv.replace("[", "").replace("]", "")
    conv = conv.replace("'", "").replace('"', "")
    
    ids = conv.split()

    # 🔥 IMPORTANT FIX
    ids = [i.strip() for i in ids]

    for i in range(len(ids) - 1):
        input_line = line_dict.get(ids[i])
        target_line = line_dict.get(ids[i+1])

        if input_line is not None and target_line is not None:
            pairs.append((input_line, target_line))

print("Total pairs:", len(pairs))

Total pairs: 207370


In [38]:
# checking pairing of data
print(ids[:5])
print(list(line_dict.keys())[:5])

['L666520', 'L666521', 'L666522']
['L1045', 'L1044', 'L985', 'L984', 'L925']


In [39]:
# clean text
# We use re to clean and standardize text before processing.
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)
    return text

pairs = [(clean_text(i), clean_text(o)) for i, o in pairs]

In [40]:
# reduce dataset
pairs = pairs[:10000]

In [41]:
# create dataframe
df = pd.DataFrame(pairs, columns=["input", "response"])

In [42]:
# build vocabulary
corpus = list(df["input"]) + list(df["response"])

vocab = set()

for text in corpus:
    for word in text.split():
        vocab.add(word)

vocab = list(vocab)

In [43]:
# tf
def tf(text):
    words = text.split()
    vec = np.zeros(len(vocab))
    
    for word in words:
        if word in vocab:
            vec[vocab.index(word)] += 1
    
    if len(words) > 0:
        vec = vec / len(words)
    
    return vec

In [44]:
# idf
idf = np.zeros(len(vocab))
N = len(corpus)

for i, word in enumerate(vocab):
    count = 0
    for text in corpus:
        if word in text.split():
            count += 1
    
    idf[i] = np.log((N + 1) / (count + 1)) + 1

In [45]:
# tf-idf
def tfidf(text):
    return tf(text) * idf

In [46]:
# cosine similarity
def cosine_similarity(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [47]:
# chatbot function
def chatbot_reply(user_input):
    user_input = clean_text(user_input)
    user_vec = tfidf(user_input)

    best_score = -1
    best_response = ""

    for i in range(len(df)):
        input_vec = tfidf(df.iloc[i]["input"])
        score = cosine_similarity(user_vec, input_vec)

        if score > best_score:
            best_score = score
            best_response = df.iloc[i]["response"]

    return best_response

In [ ]:
# run chatbot
print("Chatbot is ready! Type 'exit' to stop.")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break

    print("Bot:", chatbot_reply(user))